# rotation-matrix-3d-y-axis — ex3: rotate a single 3-vector

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `rotation-matrix-3d-y-axis`. When a test cell passes, your progress is reported back to your account.

**What you'll practice.** Five rotation-matrix patterns that ramp from `cos/sin` → matrix assembly → rotate-a-vector → composition law → rotate-a-batch. Read the docstring, fill the function body, run the test cell. The solution sits in the collapsed `<details>` block below each exercise.

**Per-exercise structure** (Doughty et al. ACE 2024 — `[Bloom level] + [LO] + [Keywords] + [KCs]`):
Each exercise begins with a yaml block stating its Bloom cognitive level, learning objective, keywords, and the knowledge components (KCs) it targets. This makes the cognitive demand explicit instead of buried.

## Setup

In [ ]:
import math
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Applied patterns and advanced` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `rotation-matrix-3d-y-axis`**, which bridges to the bank subtopic `Numpy: Applied patterns and advanced` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "rotation-matrix-3d-y-axis"
DD_SUBTOPIC = "Numpy: Applied patterns and advanced"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Y-axis rotation — quick refresher

**The matrix.** For a right-hand rotation by `θ` about the Y axis:
```
R_y(θ) = [[ cos θ,  0,  sin θ],
          [ 0,      1,  0    ],
          [-sin θ,  0,  cos θ]]
```

**Why the middle row is `[0, 1, 0]`.** The Y axis is the rotation axis — anything along Y stays where it is. The X-Z plane is what gets rotated.

**Right-hand convention.** Looking down the +Y axis, the rotation goes counter-clockwise: +X → -Z, +Z → +X.

**Acting on vectors.**
- Column-vector: `v' = R @ v` (input shape `(3,)`).
- Batch of row-vectors: `points' = points @ R.T` (input shape `(N, 3)`).

**Composition.** `R_y(α) @ R_y(β) = R_y(α + β)` — rotations about a single axis add angles.

### Exercise 3 — rotate a single 3-vector

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply a rotation matrix to a 3-vector via matrix-vector multiplication.
> Keywords: matvec, right-hand-rule, geometric-check
> ```

**KCs targeted:** `rotation-applied-to-vector`

Implement `ex3_rotate_vector(v, theta)` to rotate the 3-vector `v` by angle `theta` about the Y axis. Output shape `(3,)`.

Use `R_y(θ) @ v` where you build `R_y` from Exercise 2.

Geometric check (right-hand rule, looking down +Y):
- `(1, 0, 0)` rotated by π/2 → `(0, 0, -1)` (X goes to -Z).
- `(0, 0, 1)` rotated by π/2 → `(1, 0, 0)` (+Z goes to +X).
- Any `(0, y, 0)` stays fixed.

In [ ]:
def ex3_rotate_vector(v: Tensor, theta: Tensor) -> Tensor:
    """Rotate v (shape (3,)) about Y by theta. Returns (3,)."""
    raise NotImplementedError()


def _test_ex3():
    import math
    half_pi = t.tensor(math.pi / 2)
    # +X axis rotates to -Z under right-hand rule.
    out = ex3_rotate_vector(t.tensor([1.0, 0.0, 0.0]), half_pi)
    assert out.shape == (3,), f'shape mismatch: {out.shape}'
    assert t.allclose(out, t.tensor([0.0, 0.0, -1.0]), atol=1e-6), f'(1,0,0) @ π/2 should be (0,0,-1), got {out.tolist()}'
    # +Z rotates to +X.
    out2 = ex3_rotate_vector(t.tensor([0.0, 0.0, 1.0]), half_pi)
    assert t.allclose(out2, t.tensor([1.0, 0.0, 0.0]), atol=1e-6), f'(0,0,1) @ π/2 should be (1,0,0), got {out2.tolist()}'
    # Y-component fixed.
    out3 = ex3_rotate_vector(t.tensor([0.0, 7.0, 0.0]), half_pi)
    assert t.allclose(out3, t.tensor([0.0, 7.0, 0.0]), atol=1e-6), 'pure Y vector must be fixed by Y rotation'
    # theta = 0 is identity.
    out4 = ex3_rotate_vector(t.tensor([1.0, 2.0, 3.0]), t.tensor(0.0))
    assert t.allclose(out4, t.tensor([1.0, 2.0, 3.0]), atol=1e-6), 'theta=0 must be identity'
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_rotate_vector(v: Tensor, theta: Tensor) -> Tensor:
    c = t.cos(theta).item()
    s = t.sin(theta).item()
    R = t.tensor([
        [c, 0.0, s],
        [0.0, 1.0, 0.0],
        [-s, 0.0, c],
    ])
    return R @ v
```

**Why `R @ v` not `v @ R`?** Conventional rotation matrices act on *column* vectors: `v_rotated = R @ v_column`. If you store vectors as rows (Numpy / PyTorch default), you'd write `v_rotated = v_row @ R.T`. Same math — just transpose the matrix to match your vector layout.

**Length preservation.** Rotation matrices are orthogonal (R @ R.T = I), so `||R @ v|| == ||v||` — a great sanity check during debugging. If your output norm changes, you almost certainly have a sign error or an unnormalised axis.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()